<a href="https://colab.research.google.com/github/eshwarjyEnder-Cyber/Ender/blob/main/SLM_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import random
import re
from collections import Counter
from transformers import pipeline

# Set device to GPU if available in Colab, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

Using device: cuda



In [3]:
# ==========================================
# STEP 2: Simulate Real Data
# ==========================================
print("Generating 1,000 synthetic movie reviews...")
positive_keywords = ["great", "awesome", "fantastic", "loved", "excellent", "amazing", "beautiful", "brilliant", "perfect", "enjoyed"]
negative_keywords = ["terrible", "awful", "bad", "hated", "worst", "boring", "dull", "poor", "garbage", "waste"]
neutral_fillers   = ["the movie was", "i thought it was", "the acting was", "overall it was", "the plot was", "the director made it"]

data = []
for _ in range(500):
    # Generate Positive Reviews (Label 1)
    pos_review = f"{random.choice(neutral_fillers)} {random.choice(positive_keywords)} and {random.choice(positive_keywords)}"
    data.append({"text": pos_review, "label": 1})

    # Generate Negative Reviews (Label 0)
    neg_review = f"{random.choice(neutral_fillers)} {random.choice(negative_keywords)} and {random.choice(negative_keywords)}"
    data.append({"text": neg_review, "label": 0})

# Shuffle the dataset
random.shuffle(data)
df = pd.DataFrame(data)

Generating 1,000 synthetic movie reviews...


In [4]:
df

,text,label
0,the director made it terrible and bad,0
1,the plot was perfect and fantastic,1
2,the director made it poor and bad,0
3,the plot was hated and dull,0
4,overall it was great and great,1
...,...,...
995,the director made it great and amazing,1
996,overall it was amazing and loved,1
997,overall it was worst and worst,0
998,the movie was garbage and poor,0


In [5]:
# ==========================================
# STEP 3: Data Preprocessing & Tokenization
# ==========================================
print("Preprocessing and Tokenizing data...")

# Clean Text (lowercase and remove punctuation)
def clean_text(text):
    text = text.lower()
    return re.sub(r'[^a-z\s]', '', text)

df['clean_text'] = df['text'].apply(clean_text)

# Build a Vocabulary
all_words = ' '.join(df['clean_text']).split()
vocab_counts = Counter(all_words)
# Map words to IDs (start from 2 to reserve 0 for PAD and 1 for UNK)
vocab = {word: i+2 for i, (word, count) in enumerate(vocab_counts.items())}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1


# Convert Sentence to Sentence IDs
def tokenize(text, max_length=10):
    tokens = [vocab.get(word, vocab['<UNK>']) for word in text.split()]
    # Pad sequences so they are all the same length (required for tensors)
    if len(tokens) < max_length:
        tokens.extend([vocab['<PAD>']] * (max_length - len(tokens)))
    return tokens[:max_length]

df['tokens'] = df['clean_text'].apply(tokenize)

Preprocessing and Tokenizing data...


In [6]:
df.head()

,text,label,clean_text,tokens
0,the director made it terrible and bad,0,the director made it terrible and bad,"[2, 3, 4, 5, 6, 7, 8, 0, 0, 0]"
1,the plot was perfect and fantastic,1,the plot was perfect and fantastic,"[2, 9, 10, 11, 7, 12, 0, 0, 0, 0]"
2,the director made it poor and bad,0,the director made it poor and bad,"[2, 3, 4, 5, 13, 7, 8, 0, 0, 0]"
3,the plot was hated and dull,0,the plot was hated and dull,"[2, 9, 10, 14, 7, 15, 0, 0, 0, 0]"
4,overall it was great and great,1,overall it was great and great,"[16, 5, 10, 17, 7, 17, 0, 0, 0, 0]"


In [7]:
df.tail()

,text,label,clean_text,tokens
995,the director made it great and amazing,1,the director made it great and amazing,"[2, 3, 4, 5, 17, 7, 32, 0, 0, 0]"
996,overall it was amazing and loved,1,overall it was amazing and loved,"[16, 5, 10, 32, 7, 27, 0, 0, 0, 0]"
997,overall it was worst and worst,0,overall it was worst and worst,"[16, 5, 10, 26, 7, 26, 0, 0, 0, 0]"
998,the movie was garbage and poor,0,the movie was garbage and poor,"[2, 28, 10, 25, 7, 13, 0, 0, 0, 0]"
999,the director made it terrible and worst,0,the director made it terrible and worst,"[2, 3, 4, 5, 6, 7, 26, 0, 0, 0]"


In [8]:
# Prepare pytorch sensors
X = torch.tensor(df['tokens'].tolist(), dtype = torch.long).to(device)
y = torch.tensor(df['label'].tolist(), dtype=torch.float32).unsqueeze(1).to(device)

In [9]:
X[:1]

tensor([[2, 3, 4, 5, 6, 7, 8, 0, 0, 0]], device='cuda:0')

In [10]:
y[:1]

tensor([[0.]], device='cuda:0')

Sentence --> Token IDs --> Embeddings -->

In [12]:
# ==========================================
# STEP 4: Build the Custom SLM Architecture
# ==========================================
class CustomSLM(nn.Module) :
  def __init__(self, vocab_size,embed_dim,hidden_dim, output_dim):
    super(CustomSLM,self).__init__()
    self.embedding = nn.Embedding(vocab_size,embed_dim,padding_idx=0)
    self.fc1  = nn.Linear(embed_dim, hidden_dim)
    self.relu = nn.ReLU()
    self.fc2  = nn.Linear(hidden_dim,output_dim)
    self.sigmoid = nn.Sigmoid()

  def forward(self, text):
    embedded = self.embedding(text)
    # average
    pooled = embedded.mean(dim=1)
    hidden = self.relu(self.fc1(pooled))
    output = self.sigmoid(self.fc2(hidden))
    return output

In [13]:
VOCAB_SIZE = len(vocab)
model = CustomSLM(vocab_size = VOCAB_SIZE, embed_dim = 16, hidden_dim = 32,
                  output_dim=1).to(device)

In [14]:
# ==========================================
# STEP 5: Train the Custom Model
# ==========================================
print("\nTraining the Custom SLM from scratch...")
criterion = nn.BCELoss() # Binary Cross Entropy for 0/1 classification
optimizer = optim.Adam(model.parameters(), lr=0.01)
epochs = 15

for epoch in range(epochs):
  model.train()
  optimizer.zero_grad()

  predictions = model(X)
  loss = criterion(predictions, y)
  loss.backward()
  optimizer.step()

  if (epoch+1) %3 == 0 :
    # Accuracy (basic)
    predicted_classes = (predictions > 0.5).float()
    accuracy = (predicted_classes == y).sum().item()/len(y)
    print(f"Epoch: {epoch+1:02d}/{epochs} | Loss: {loss.item():.4f} | Accuracy: {accuracy*100:.1f}%")

print("Training Completed! \n")


Training the Custom SLM from scratch...
Epoch: 03/15 | Loss: 0.6750 | Accuracy: 78.0%
Epoch: 06/15 | Loss: 0.6445 | Accuracy: 86.6%
Epoch: 09/15 | Loss: 0.5972 | Accuracy: 87.9%
Epoch: 12/15 | Loss: 0.5301 | Accuracy: 89.5%
Epoch: 15/15 | Loss: 0.4441 | Accuracy: 93.7%
Training Completed! 



In [15]:
# ============================
# Test the Custom SLM
# ============================

# test_sentence = "This movie was absolutely fantastic"
test_sentence = "This was the worst movie I have ever seen"

# Preprocess
clean = clean_text(test_sentence)

# Tokenize
tokens = tokenize(clean)

# Convert to Tensor
input_tensor = torch.tensor([tokens], dtype=torch.long).to(device)

# Prediction
model.eval()
with torch.no_grad():
    prediction = model(input_tensor)

probability = prediction.item()

print(f"Sentence : {test_sentence}")
print(f"Probability of Positive : {probability:.4f}")

if probability >= 0.5:
    print("Prediction : Positive")
else:
    print("Prediction : Negative")

Sentence : This was the worst movie I have ever seen
Probability of Positive : 0.3688
Prediction : Negative


In [16]:
# ==========================================
# STEP 6: Compare with Pre-Trained DistilBERT
# ==========================================
print("-" * 50)
print("Evaluating Sample Sentences")
print("-" * 50)

# Sample sentences to test both models
test_sentences = [
    "the movie was absolutely fantastic and brilliant",
    "i thought it was terrible and a waste",
    "it was so good that it cured my insomnia",
    "What a fantastic way to ruin my day"
]

# 1. Test Custom Model
print(">>> Custom SLM Predictions:")
model.eval()
with torch.no_grad():
    for sentence in test_sentences:
      clean_s = clean_text(sentence)
      tokens = tokenize(clean_s)
      tensor_input = torch.tensor([tokens], dtype=torch.long).to(device)
      prediction = model(tensor_input).item()
      sentiment = "Positive" if prediction > 0.5 else "Negative"
      print(f"Text: '{sentence}' \n-> Sentiment: {sentiment} (Score: {prediction:.4f})\n")

--------------------------------------------------
Evaluating Sample Sentences
--------------------------------------------------
>>> Custom SLM Predictions:
Text: 'the movie was absolutely fantastic and brilliant' 
-> Sentiment: Positive (Score: 0.5025)

Text: 'i thought it was terrible and a waste' 
-> Sentiment: Negative (Score: 0.2748)

Text: 'it was so good that it cured my insomnia' 
-> Sentiment: Positive (Score: 0.5621)

Text: 'What a fantastic way to ruin my day' 
-> Sentiment: Positive (Score: 0.5601)



In [17]:
# 2. Test DistilBERT (Pre-Trained)
print(">>> DistilBERT Predictions (Downloading model if first run...):")
pipe_device = 0 if torch.cuda.is_available() else -1
classifier = pipeline("sentiment-analysis",
                      model="distilbert-base-uncased-finetuned-sst-2-english",
                      device=pipe_device)

>>> DistilBERT Predictions (Downloading model if first run...):


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [18]:
results = classifier(test_sentences)
for sentence, result in zip(test_sentences, results):
    print(f"Text: '{sentence}' \n-> Sentiment: {result['label']} (Confidence: {result['score']:.4f})\n")

Text: 'the movie was absolutely fantastic and brilliant' 
-> Sentiment: POSITIVE (Confidence: 0.9999)

Text: 'i thought it was terrible and a waste' 
-> Sentiment: NEGATIVE (Confidence: 0.9998)

Text: 'it was so good that it cured my insomnia' 
-> Sentiment: POSITIVE (Confidence: 0.9997)

Text: 'What a fantastic way to ruin my day' 
-> Sentiment: NEGATIVE (Confidence: 0.9938)

